# L24 drift versus naive-reuse K=2 absolute GT loss

This production analysis tests whether delta-t=1 and delta-t=2 change in the base full-L24 backbone/neck representation is associated with absolute ground-truth error from deployed RepDETR3D EVA-02 naive reuse K=2. It is an association analysis, not a causal reuse-error decomposition.

The normal path requires Task 2 collector artifacts. Set `TM_L24_REUSE_RISK_SMOKE=1` only for a deterministic synthetic execution smoke test; smoke mode never writes paper figures.

In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
from sklearn.isotonic import IsotonicRegression

WORKING_DIR = Path.cwd().resolve()
if (WORKING_DIR / 'tools').is_dir():
    REPO_ROOT = WORKING_DIR
elif (WORKING_DIR.parent / 'tools').is_dir():
    REPO_ROOT = WORKING_DIR.parent
else:
    raise RuntimeError('execute this notebook from the repository root or notebooks directory')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tools.l24_drift_reuse_risk import (
    build_panel_summary,
    filter_predictions_by_score,
    frame_error,
    validate_collection_manifest,
    validate_target_tables,
    zero_free_data_limits,
)

SYNTHETIC_SMOKE = os.environ.get('TM_L24_REUSE_RISK_SMOKE') == '1'
EXPECTED_SCENES = 100
EXPECTED_TARGETS_PER_SCENE = 20
EXPECTED_TARGETS = EXPECTED_SCENES * EXPECTED_TARGETS_PER_SCENE
BOOTSTRAP_REPETITIONS = 20 if SYNTHETIC_SMOKE else 2000
SEED = 20260805
MANIFEST_PATH = Path(os.environ.get(
    'TM_L24_REUSE_RISK_MANIFEST',
    REPO_ROOT / 'outputs/l24_drift_reuse_risk/collection_manifest.json',
))
OUTPUT_DIR = Path(os.environ.get(
    'TM_L24_REUSE_RISK_OUTPUT_DIR',
    REPO_ROOT / ('outputs/l24_drift_reuse_risk_smoke' if SYNTHETIC_SMOKE else 'outputs/l24_drift_reuse_risk'),
))
PAPER_DIR = REPO_ROOT / 'paper/figures'
DATA_ROOT = os.environ.get('TM_L24_REUSE_RISK_DATA_ROOT', str(REPO_ROOT / 'data/nuscenes'))
NUSC_VERSION_OVERRIDE = os.environ.get('TM_L24_REUSE_RISK_NUSC_VERSION')
NUSC_VERSION = 'synthetic-smoke' if SYNTHETIC_SMOKE else None
SCORE_FLOOR = 0.10
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not SYNTHETIC_SMOKE and BOOTSTRAP_REPETITIONS != 2000:
    raise RuntimeError('production analysis requires exactly 2,000 bootstrap repetitions')
assert EXPECTED_TARGETS == 2000


In [ ]:
NUSCENES_TO_DETECTION = {
    'vehicle.car': 'car', 'vehicle.truck': 'truck',
    'vehicle.bus.bendy': 'bus', 'vehicle.bus.rigid': 'bus',
    'vehicle.construction': 'construction_vehicle',
    'vehicle.motorcycle': 'motorcycle', 'vehicle.bicycle': 'bicycle',
    'vehicle.trailer': 'trailer',
    'human.pedestrian.adult': 'pedestrian',
    'human.pedestrian.child': 'pedestrian',
    'human.pedestrian.construction_worker': 'pedestrian',
    'human.pedestrian.police_officer': 'pedestrian',
    'movable_object.trafficcone': 'traffic_cone',
    'movable_object.barrier': 'barrier',
}

def normalise_detection(box, is_prediction):
    score = box.get('detection_score', box.get('score', 1.0)) if is_prediction else 1.0
    velocity = np.nan_to_num(np.asarray(box.get('velocity', [0.0, 0.0]), dtype=float))
    return {
        'name': box['detection_name'] if 'detection_name' in box else box['name'],
        'translation': np.asarray(box['translation'], dtype=float),
        'size': np.asarray(box['size'], dtype=float),
        'rotation': np.asarray(box['rotation'], dtype=float),
        'velocity': velocity[:2],
        'score': float(score),
    }

def load_ground_truth(data_root, version, tokens):
    from nuscenes.nuscenes import NuScenes

    nusc = NuScenes(version=version, dataroot=str(data_root), verbose=False)
    ground_truth = {}
    for token in tokens:
        sample = nusc.get('sample', token)
        boxes = []
        for annotation_token in sample['anns']:
            annotation = nusc.get('sample_annotation', annotation_token)
            name = NUSCENES_TO_DETECTION.get(annotation['category_name'])
            if name is None:
                continue
            boxes.append(normalise_detection({
                'name': name,
                'translation': annotation['translation'],
                'size': annotation['size'],
                'rotation': annotation['rotation'],
                'velocity': nusc.box_velocity(annotation_token),
            }, is_prediction=False))
        ground_truth[token] = boxes
    return ground_truth


In [ ]:
def synthetic_inputs():
    rng = np.random.default_rng(SEED)
    rows, drift_rows, predictions, ground_truth = [], [], {}, {}
    for scene_index in range(EXPECTED_SCENES):
        scene_token = f'smoke-scene-{scene_index:03d}'
        for frame_index in range(EXPECTED_TARGETS_PER_SCENE):
            token = f'{scene_token}-target-{frame_index:02d}'
            drift_dt1 = 0.01 + 0.98 * (scene_index * EXPECTED_TARGETS_PER_SCENE + frame_index) / EXPECTED_TARGETS
            drift_dt2 = min(0.99, drift_dt1 + 0.02)
            target = {
                'name': 'car', 'translation': [0.0, 0.0, 0.0],
                'size': [2.0, 4.0, 1.5], 'rotation': [1.0, 0.0, 0.0, 0.0],
                'velocity': [0.0, 0.0], 'score': 1.0,
            }
            error_scale = 0.05 + 0.65 * drift_dt1 + rng.normal(0.0, 0.015)
            prediction = {**target, 'translation': [max(0.0, error_scale), 0.0, 0.0],
                          'score': max(0.05, 0.98 - error_scale)}
            rows.append({'token': token, 'scene_token': scene_token})
            drift_rows.append({'token': token, 'scene_token': scene_token,
                               'drift_dt1': drift_dt1, 'drift_dt2': drift_dt2})
            predictions[token] = [prediction]
            ground_truth[token] = [target]
    return pd.DataFrame(rows), pd.DataFrame(drift_rows), predictions, ground_truth

if SYNTHETIC_SMOKE:
    selection, drift, predictions, ground_truth = synthetic_inputs()
    manifest = {'run_id': 'synthetic-smoke', 'selected_target_count': EXPECTED_TARGETS,
                'qualified_scene_count': EXPECTED_SCENES}
else:
    if not MANIFEST_PATH.is_file():
        raise FileNotFoundError(
            f'missing collector artifact: {MANIFEST_PATH}. Run bash tools/run_l24_drift_reuse_risk.sh first.')
    manifest = json.loads(MANIFEST_PATH.read_text())
    artifact_paths = validate_collection_manifest(manifest)
    if NUSC_VERSION_OVERRIDE and NUSC_VERSION_OVERRIDE != manifest['dataset_version']:
        raise ValueError('TM_L24_REUSE_RISK_NUSC_VERSION must match the collected dataset_version')
    NUSC_VERSION = manifest['dataset_version']
    selection = pd.read_csv(artifact_paths['selection'])
    drift = pd.read_csv(artifact_paths['drift'])
    prediction_payload = json.loads(artifact_paths['predictions'].read_text())
    predictions = prediction_payload.get('results')
    if not isinstance(predictions, dict):
        raise ValueError('results_nusc.json must contain a results mapping')

per_frame = validate_target_tables(selection, drift, predictions)
filtered_predictions = {
    token: filter_predictions_by_score(records, score_floor=SCORE_FLOOR)
    for token, records in predictions.items()
}
# Production GT loading deliberately happens only after artifact and exact-token validation.
if not SYNTHETIC_SMOKE:
    ground_truth = load_ground_truth(DATA_ROOT, NUSC_VERSION, per_frame.token.tolist())

per_frame['prediction_loss'] = [
    frame_error([normalise_detection(box, True) for box in filtered_predictions[token]], ground_truth[token], True)
    for token in per_frame.token
]
per_frame['box_loss'] = [
    frame_error([normalise_detection(box, True) for box in filtered_predictions[token]], ground_truth[token], False)
    for token in per_frame.token
]
if not np.isfinite(per_frame[['prediction_loss', 'box_loss']].to_numpy(float)).all():
    raise ValueError('GT loss calculation produced non-finite values')

quantile_summary, correlation_summary = build_panel_summary(
    per_frame, repetitions=BOOTSTRAP_REPETITIONS, seed=SEED)
per_frame.head(), correlation_summary


In [ ]:
PANEL_ORDER = [('prediction_loss', 'Prediction loss'), ('box_loss', 'Box loss')]
OFFSET_ORDER = [('dt1', 'delta-t=1'), ('dt2', 'delta-t=2')]
QUINTILE_COLOR = '#0072B2'  # Okabe-Ito blue
AXIS_PADDING_FRACTION = 0.05

def density_values(x, y):
    if len(x) < 3 or np.ptp(x) == 0 or np.ptp(y) == 0:
        return np.zeros(len(x))
    try:
        return gaussian_kde(np.vstack([x, y]))(np.vstack([x, y]))
    except np.linalg.LinAlgError:
        return np.zeros(len(x))

def render_figure(frame, quantiles, correlations, output_stem):
    axis_limits = []
    figure, axes = plt.subplots(2, 2, figsize=(10.2, 7.2), constrained_layout=True)
    for row, (outcome, outcome_label) in enumerate(PANEL_ORDER):
        for column, (offset, offset_label) in enumerate(OFFSET_ORDER):
            axis = axes[row, column]
            drift_column = f'drift_{offset}'
            x = frame[drift_column].to_numpy(float)
            y = frame[outcome].to_numpy(float)
            order = np.argsort(density_values(x, y))
            axis.scatter(x[order], y[order], c=density_values(x, y)[order], cmap='viridis',
                         s=9, alpha=0.55, linewidths=0, rasterized=True)
            trend = IsotonicRegression(increasing='auto', out_of_bounds='clip').fit(x, y)
            trend_x = np.linspace(x.min(), x.max(), 200)
            axis.plot(trend_x, trend.predict(trend_x), color='black', lw=1.5, label='isotonic trend')
            panel_quantiles = quantiles.query('offset == @offset and outcome == @outcome')
            axis.errorbar(panel_quantiles.drift_mean, panel_quantiles.outcome_mean,
                          yerr=[panel_quantiles.outcome_mean - panel_quantiles.ci_low,
                                panel_quantiles.ci_high - panel_quantiles.outcome_mean],
                          fmt='o', color=QUINTILE_COLOR, ecolor=QUINTILE_COLOR,
                          capsize=3, ms=4, label='quintile mean, 95% CI')
            correlation = correlations.query('offset == @offset and outcome == @outcome').iloc[0]
            axis.text(0.03, 0.97,
                      f"Spearman rho={correlation.spearman:.3f} [{correlation.ci_low:.3f}, {correlation.ci_high:.3f}]",
                      transform=axis.transAxes, va='top', fontsize=9,
                      bbox={'facecolor': 'white', 'alpha': 0.82, 'edgecolor': 'none'})
            x_limits = zero_free_data_limits(x, padding_fraction=AXIS_PADDING_FRACTION)
            y_limits = zero_free_data_limits(y, padding_fraction=AXIS_PADDING_FRACTION)
            axis.set_xlim(*x_limits)
            axis.set_ylim(*y_limits)
            axis_limits.append({'outcome': outcome, 'offset': offset,
                                'x_limits': list(x_limits), 'y_limits': list(y_limits)})
            axis.set_title(f'{outcome_label}, {offset_label}')
            axis.set_xlabel('Full-L24 cosine drift')
            axis.set_ylabel('Absolute GT error')
    figure.savefig(output_stem.with_suffix('.pdf'), bbox_inches='tight')
    figure.savefig(output_stem.with_suffix('.png'), dpi=300, bbox_inches='tight')
    plt.close(figure)
    return axis_limits

figure_stem = OUTPUT_DIR / 'fig_l24_drift_vs_naive_reuse_gt_loss'
figure_axis_limits = render_figure(per_frame, quantile_summary, correlation_summary, figure_stem)
if not SYNTHETIC_SMOKE:
    PAPER_DIR.mkdir(parents=True, exist_ok=True)
    for suffix in ('.pdf', '.png'):
        shutil.copy2(figure_stem.with_suffix(suffix),
                     PAPER_DIR / f'fig_l24_drift_vs_naive_reuse_gt_loss{suffix}')


In [ ]:
per_frame.to_csv(OUTPUT_DIR / 'per_frame_gt_loss.csv', index=False)
quantile_summary.to_csv(OUTPUT_DIR / 'quantile_summary.csv', index=False)
correlation_summary.to_csv(OUTPUT_DIR / 'correlation_summary.csv', index=False)
analysis_config = {
    'synthetic_smoke': SYNTHETIC_SMOKE,
    'manifest_path': str(MANIFEST_PATH.resolve()),
    'output_dir': str(OUTPUT_DIR.resolve()),
    'bootstrap_repetitions': BOOTSTRAP_REPETITIONS,
    'seed': SEED,
    'expected_scenes': EXPECTED_SCENES,
    'expected_targets': EXPECTED_TARGETS,
    'data_root': DATA_ROOT,
    'nuscenes_version': NUSC_VERSION,
    'score_floor': SCORE_FLOOR,
    'axis_padding_fraction': AXIS_PADDING_FRACTION,
}
(OUTPUT_DIR / 'analysis_config.json').write_text(json.dumps(analysis_config, indent=2, sort_keys=True))
validation_summary = {
    'run_id': manifest['run_id'],
    'synthetic_smoke': SYNTHETIC_SMOKE,
    'frames': int(len(per_frame)),
    'scenes': int(per_frame.scene_token.nunique()),
    'correlation_rows': int(len(correlation_summary)),
    'expected_frames': EXPECTED_TARGETS,
    'expected_scenes': EXPECTED_SCENES,
    'figure_axis_limits': figure_axis_limits,
}
(OUTPUT_DIR / 'validation_summary.json').write_text(json.dumps(validation_summary, indent=2, sort_keys=True))
required = [
    OUTPUT_DIR / 'per_frame_gt_loss.csv', OUTPUT_DIR / 'quantile_summary.csv',
    OUTPUT_DIR / 'correlation_summary.csv', OUTPUT_DIR / 'analysis_config.json',
    OUTPUT_DIR / 'validation_summary.json', figure_stem.with_suffix('.pdf'),
    figure_stem.with_suffix('.png'),
]
if not SYNTHETIC_SMOKE:
    required += [PAPER_DIR / 'fig_l24_drift_vs_naive_reuse_gt_loss.pdf',
                 PAPER_DIR / 'fig_l24_drift_vs_naive_reuse_gt_loss.png']
missing = [str(path) for path in required if not path.is_file() or path.stat().st_size == 0]
if missing:
    raise RuntimeError(f'missing or empty required outputs: {missing}')
if len(per_frame) != EXPECTED_TARGETS or per_frame.scene_token.nunique() != EXPECTED_SCENES:
    raise RuntimeError('analysis did not retain the exact 2,000-target / 100-scene study')
if len(correlation_summary) != 4:
    raise RuntimeError('analysis did not generate all four panel summaries')
if any(limits['x_limits'][0] <= 0.0 or limits['y_limits'][0] <= 0.0
       for limits in figure_axis_limits):
    raise RuntimeError('figure axis limits must remain informative and zero-free')
print(f"Validated {len(per_frame)} targets across {per_frame.scene_token.nunique()} scenes; wrote {len(required)} artifacts.")
